In [ ]:
from google.colab import files
uploaded=files.upload()
file_name=list(uploaded.keys())[0]

TypeError: 'NoneType' object is not subscriptable

In [ ]:
from google.colab import files
uploaded=files.upload()
file_name=list(uploaded.keys())[0]

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report
)
# FIX 1: Use the file_name variable obtained from the file upload
df=pd.read_excel(file_name)
print("dataset loaded succesfully!\n")
print(df.head())
# FIX 3: Correcting a typo 'colums' to 'columns'
df.columns=df.columns.str.strip()
df.rename(columns={
    # FIX 2: Use a raw string to avoid SyntaxWarning for '\k'
    "Soil Nitrogen (mg/kg)":"soil_nitrogen", # Corrected key to match actual column name
    "Avg Rainfall (mm)":"rainfall" # Corrected key to match actual column name
}, inplace=True)
if "sample ID" in df.columns:
  df.drop(columns=['sample ID'],inplace=True)

if 'class label' in df.columns and 'class' in df.columns:
  df.drop(columns=['class'],inplace=True)

if 'class label' in df.columns:
  df.rename(columns={"class label":"class"},inplace=True)

# FIX 4: Correcting a typo 'soi_nitrogen' to 'soil_nitrogen'
x=df[["soil_nitrogen","rainfall"]]
y=df["class"]

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.3,random_state=42)

def train_and_plot(c,subplot_index):
   model=svm.SVC(kernel='linear',c=c)
   model.fit(x_train,y_train)

   # FIX 5: Correcting a typo 'prdict' to 'predict'
   y_pred=model.predict(x_test)

   print(f"\n====results for c={c}====")
   print("accuracy:",accuracy_score(y_test,y_pred))
   # FIX 6: The confusion matrix was using y_train instead of y_pred
   print("confusion matrix:\n",confusion_matrix(y_test,y_pred))
   print("classification report:\n",classification_report(y_test,y_pred))

   plt.subplot(1,3,subplot_index)

   for label,color in zip([1,-1],['green','red']):
       subset=df[df['class']==label]
       plt.scatter(subset['soil_nitrogen'],
                   subset["rainfall"],
                   c=color,
                   label=("healthy" if label==1 else "stressed"))
   ax=plt.gca()
   xlim=ax.get_xlim()
   ylim=ax.get_ylim()

   xx=np.linspace(xlim[0],xlim[1],30)
   yy=np.linspace(ylim[0],ylim[1],30)
   yy,xx=np.meshgrid(yy,xx)

   xy=np.vstack([xx.ravel(),yy.ravel()]).T
   z=model.decision_function(xy).reshape(xx.shape)

   ax.contour(xx,yy,z,levels=[0], linewidths=2)
   ax.contour(xx,yy,z,levels=[-1,1],linestyles=['--','--'])
   ax.scatter(model.support_vectors_[:,0],
             model.support_vectors_[:,1],
             s=100,facecolors='none',
             edgecolors='black',
             label='support vectors')
   plt.xlabel("soil nitrogen")
   plt.ylabel("rainfall")
   plt.title(f"c={c}")
   plt.legend()

# FIX 7: Move plotting setup outside the function
plt.figure(figsize=(18,5))
c_values=[0.01,1,100]
for i,c in enumerate(c_values):
    train_and_plot(c,i+1)

plt.suptitle("SVM:soft margin vs optimal vs hard margin")
plt.tight_layout()
plt.show()